# LLM Fundamentals - Hands-On Tour

A single notebook walking through the core ideas from `../docs/`,
end to end: one API call &rarr; tokens &rarr; sampling &rarr; memory &rarr;
embeddings &rarr; a minimal RAG pipeline. Framed the way I actually think
about this stuff, as a platform/DevOps engineer - every section calls
out the infra concept it maps to.

## Setup

```bash
ollama pull llama3.1:8b
ollama pull nomic-embed-text
pip install openai tiktoken numpy
```

Everything below runs against a **local Ollama model** - no API key,
no cost. See the last cell for how to point this at a hosted API
instead.

In [ ]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
MODEL = "llama3.1:8b"

## 1. The basic call

Recall from `docs/03-How-LLMs-Work.md`: this isn't a database lookup,
it's the model predicting the most statistically plausible continuation
of your prompt, one token at a time.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "In one sentence, what does a load balancer do?"}],
)
print(response.choices[0].message.content)
print()
print(f"prompt tokens: {response.usage.prompt_tokens}")
print(f"completion tokens: {response.usage.completion_tokens}")

## 2. Tokens are the real unit of cost

See `Tokenization.ipynb` for the deep dive - here's the quick version,
tied to text you'd actually paste into a prompt.

In [ ]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4")
text = "kubectl rollout restart deployment/web-app --namespace production"
print(f"'{text}'")
print(f"-> {len(enc.encode(text))} tokens")

## 3. Temperature: the "jitter" dial

`temperature=0` should be close to deterministic - good for anything a
script depends on parsing. Higher values inject more randomness - good
for brainstorming, bad for automation. See `docs/10-Temperature-TopP-and-Sampling.md`.

In [ ]:
prompt = "Write a one-line commit message for fixing a flaky integration test."

for temp in [0.0, 1.2]:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temp,
    )
    print(f"temperature={temp}: {response.choices[0].message.content.strip()}")

## 4. The model has no memory - your code does

Every call is stateless (`docs/08-Training-vs-Inference.md`). A
"conversation" is just your code resending the whole history each
turn. Watch what happens if we *don't* do that.

In [ ]:
# Without history: the model has no idea what "it" refers to.
r1 = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "My favorite tool is Terraform."}],
)
r2 = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Why do I like it?"}],  # no history!
)
print("Without history:", r2.choices[0].message.content[:150])

In [ ]:
# With history: we resend everything, so context carries over.
history = [{"role": "user", "content": "My favorite tool is Terraform."}]
r1 = client.chat.completions.create(model=MODEL, messages=history)
history.append({"role": "assistant", "content": r1.choices[0].message.content})
history.append({"role": "user", "content": "Why do I like it?"})

r2 = client.chat.completions.create(model=MODEL, messages=history)
print("With history:", r2.choices[0].message.content[:150])

## 5. Embeddings: search by meaning, not exact words

See `docs/05-Embeddings-Basics.md`. This is a different kind of index -
closer to semantic search than to `grep`. Two runbook titles below
share *zero* words but should still score as closely related.

In [ ]:
import numpy as np

def embed(text: str):
    resp = client.embeddings.create(model="nomic-embed-text", input=text)
    return np.array(resp.data[0].embedding)

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

a = embed("The pod is stuck in CrashLoopBackOff")
b = embed("Container keeps restarting and failing health checks")
c = embed("Deploy the new frontend release to staging")

print(f"a vs b (related):   {cosine_similarity(a, b):.3f}")
print(f"a vs c (unrelated):  {cosine_similarity(a, c):.3f}")

## 6. A minimal RAG pipeline

See `docs/14-Fine-Tuning-vs-RAG.md`. Instead of trusting whatever the
model memorized during training, we retrieve the *actual* relevant
runbook and hand it to the model as grounding - directly mitigating the
hallucination problem from `docs/11-Hallucinations.md`.

In [ ]:
runbooks = {
    "pod-crashloop.md": "To debug CrashLoopBackOff, run kubectl describe pod and check kubectl logs --previous for the crash reason.",
    "disk-full.md": "When disk usage hits 90%, first check /var/log for oversized log files before resizing any volume.",
    "deploy-rollback.md": "To roll back a failed deployment, run kubectl rollout undo deployment/<name>.",
}

# Index once (in a real system, cache this instead of re-embedding every run)
index = {name: embed(text) for name, text in runbooks.items()}

def retrieve(question: str, top_k: int = 1):
    q_vec = embed(question)
    scored = [(name, cosine_similarity(q_vec, vec)) for name, vec in index.items()]
    scored.sort(key=lambda x: x[1], reverse=True)
    return [runbooks[name] for name, _ in scored[:top_k]]

question = "one of our pods keeps restarting, what do I check?"
context = "\n".join(retrieve(question))
print(f"retrieved context: {context}\n")

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": f"Answer using only this context:\n{context}"},
        {"role": "user", "content": question},
    ],
    temperature=0.2,
)
print("answer:", response.choices[0].message.content)

## Wrap-up

| Section | Concept | Docs chapter |
| --- | --- | --- |
| 1 | Request/response loop | 01, 03 |
| 2 | Tokens | 04 |
| 3 | Temperature | 10 |
| 4 | Statelessness / memory | 08, 09 |
| 5 | Embeddings | 05 |
| 6 | RAG | 14 |

Everything above ran against a small local model, on purpose - no
cost, no key, safe to break. Swap in a hosted API for production-grade
quality:

```python
client = OpenAI()  # reads OPENAI_API_KEY from your environment
MODEL = "gpt-4o-mini"
```

Next: work through `docs/16-Prompt-Lifecycle.md` and
`docs/18-Best-Practices.md` to see how these pieces get wired into an
actual production pipeline with logging, validation, and guardrails.